# Pre-Entrega: Limpieza y Análisis Exploratorio de un Dataset Real
Este notebook contiene el desarrollo de una limpieza de datos rigurosa y un análisis exploratorio inicial aplicando la librería **Pandas**.

In [1]:
import pandas as pd
import numpy as np

# 1. Creación de un Dataset con desafíos técnicos (simulando datos reales de ventas)
data = {
    'ID_Transaccion': [100, 101, 102, 102, 103, 104, 105, 106, 107, 108],
    'Fecha': ['2023-01-01', '2023-01-02', '2023/01/03', '2023/01/03', '2023-01-05', '2023-01-06', None, '2023-01-08', '2023-01-09', '2023-01-10'],
    'Region': ['Norte', 'Sur', 'Este', 'Este', 'Oeste', 'Norte', 'Sur', None, 'Este', 'Oeste'],
    'Categoria': ['Electrónica', 'Hogar', 'Hogar', 'Hogar', 'Electrónica', 'Moda', 'Moda', 'Electrónica', 'Moda', None],
    'Venta_USD': ['150.50', '230.00', '45.10', '45.10', '99.99', 'no_disponible', '120.00', '340.00', '15.50', '85.00'],
    'Cantidad': [1, 2, None, None, 1, 3, 2, 5, 1, 2]
}

df_original = pd.DataFrame(data)
# Guardamos a CSV local para simular la carga con read_csv
df_original.to_csv('ventas_sucias.csv', index=False)

# Carga inicial del dataset
df = pd.read_csv('ventas_sucias.csv')
print("--- Primeras filas del dataset original ---")
display(df.head())


--- Primeras filas del dataset original ---


,ID_Transaccion,Fecha,Region,Categoria,Venta_USD,Cantidad
0,100,2023-01-01,Norte,Electrónica,150.50,1.0
1,101,2023-01-02,Sur,Hogar,230.00,2.0
2,102,2023/01/03,Este,Hogar,45.10,NaN
3,102,2023/01/03,Este,Hogar,45.10,NaN
4,103,2023-01-05,Oeste,Electrónica,99.99,1.0


## Diagnóstico del Dataset
Utilizaremos los métodos clave `.info()`, `.describe()` y `.isnull().sum()` para evaluar las inconsistencias.

In [2]:
print("--- Información General ---")
df.info()

print("\n--- Resumen Estadístico Inicial ---")
display(df.describe(include='all'))

print("\n--- Conteo de Valores Nulos ---")
print(df.isnull().sum())


--- Información General ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID_Transaccion  10 non-null     int64  
 1   Fecha           9 non-null      object 
 2   Region          9 non-null      object 
 3   Categoria       9 non-null      object 
 4   Venta_USD       10 non-null     object 
 5   Cantidad        8 non-null      float64
dtypes: float64(1), int64(1), object(4)
memory usage: 612.0+ bytes

--- Resumen Estadístico Inicial ---


,ID_Transaccion,Fecha,Region,Categoria,Venta_USD,Cantidad
count,10.00000,9,9,9,10,8.000000
unique,NaN,8,4,3,9,NaN
top,NaN,2023/01/03,Este,Electrónica,45.10,NaN
freq,NaN,2,3,3,2,NaN
mean,103.80000,NaN,NaN,NaN,NaN,2.125000
std,2.65832,NaN,NaN,NaN,NaN,1.356203
min,100.00000,NaN,NaN,NaN,NaN,1.000000
25%,102.00000,NaN,NaN,NaN,NaN,1.000000
50%,103.50000,NaN,NaN,NaN,NaN,2.000000
75%,105.75000,NaN,NaN,NaN,NaN,2.250000



--- Conteo de Valores Nulos ---
ID_Transaccion    0
Fecha             1
Region            1
Categoria         1
Venta_USD         0
Cantidad          2
dtype: int64


## Limpieza Rigurosa

Aplicaremos:
1. Eliminación de registros completamente duplicados.
2. Conversión de tipos de datos (fechas a `datetime` y ventas a `float`).
3. Imputación de nulos (moda para categorías e interpolación/media para numéricos).
4. Reseteo de índice tras eliminar registros.

In [3]:
# 1. Eliminar duplicados
print(f"Duplicados antes: {df.duplicated().sum()}")
df = df.drop_duplicates()

# 2. Corregir y convertir columna Fecha
df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce')
# Si hay fechas nulas residuales, podemos rellenarlas o eliminarlas. En este caso eliminamos filas sin fecha
df = df.dropna(subset=['Fecha'])

# 3. Corregir columna de Ventas (reemplazar valores no numéricos y pasar a float)
df['Venta_USD'] = df['Venta_USD'].replace('no_disponible', np.nan)
df['Venta_USD'] = pd.to_numeric(df['Venta_USD'], errors='coerce')

# 4. Imputar valores nulos en columnas críticas
# Imputamos 'Venta_USD' con la mediana
mediana_ventas = df['Venta_USD'].median()
df['Venta_USD'] = df['Venta_USD'].fillna(mediana_ventas)

# Imputamos 'Cantidad' con la moda (el valor más común) o un valor fijo razonable (1)
df['Cantidad'] = df['Cantidad'].fillna(1).astype(int)

# Imputamos 'Region' y 'Categoria' con 'No Especificado'
df['Region'] = df['Region'].fillna('No Especificado')
df['Categoria'] = df['Categoria'].fillna('No Especificado')

# 5. Resetear el índice para mantener la consistencia
df = df.reset_index(drop=True)

print("--- Dataset Limpio e Indexado ---")
display(df)
df.info()


Duplicados antes: 1
--- Dataset Limpio e Indexado ---


,ID_Transaccion,Fecha,Region,Categoria,Venta_USD,Cantidad
0,100,2023-01-01,Norte,Electrónica,150.500,1
1,101,2023-01-02,Sur,Hogar,230.000,2
2,103,2023-01-05,Oeste,Electrónica,99.990,1
3,104,2023-01-06,Norte,Moda,125.245,3
4,106,2023-01-08,No Especificado,Electrónica,340.000,5
5,107,2023-01-09,Este,Moda,15.500,1
6,108,2023-01-10,Oeste,No Especificado,85.000,2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID_Transaccion  7 non-null      int64         
 1   Fecha           7 non-null      datetime64[ns]
 2   Region          7 non-null      object        
 3   Categoria       7 non-null      object        
 4   Venta_USD       7 non-null      float64       
 5   Cantidad        7 non-null      int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(2)
memory usage: 468.0+ bytes


## Agregaciones de Negocio

Generamos 3 agrupaciones para responder preguntas clave sobre el desempeño del negocio:

In [4]:
# Agrupación 1: Total y promedio de ventas por Categoría de producto
ventas_por_categoria = df.groupby('Categoria').agg(
    Total_Ventas_USD=('Venta_USD', 'sum'),
    Promedio_Venta_USD=('Venta_USD', 'mean'),
    Cantidad_Articulos=('Cantidad', 'sum')
).reset_index()

print("--- 1. Ventas por Categoría ---")
display(ventas_por_categoria)

# Agrupación 2: Conteo de transacciones y ventas totales por Región
rendimiento_region = df.groupby('Region').agg(
    Transacciones=('ID_Transaccion', 'count'),
    Total_Facturado=('Venta_USD', 'sum')
).reset_index()

print("\n--- 2. Rendimiento por Región ---")
display(rendimiento_region)

# Agrupación 3: Promedio de cantidad de artículos comprados por transacción según la Categoría
cantidad_por_categoria = df.groupby('Categoria')['Cantidad'].mean().reset_index().rename(columns={'Cantidad': 'Promedio_Cantidad'})

print("\n--- 3. Promedio de Cantidad Vendida por Categoría ---")
display(cantidad_por_categoria)


--- 1. Ventas por Categoría ---


,Categoria,Total_Ventas_USD,Promedio_Venta_USD,Cantidad_Articulos
0,Electrónica,590.490,196.8300,7
1,Hogar,230.000,230.0000,2
2,Moda,140.745,70.3725,4
3,No Especificado,85.000,85.0000,2



--- 2. Rendimiento por Región ---


,Region,Transacciones,Total_Facturado
0,Este,1,15.500
1,No Especificado,1,340.000
2,Norte,2,275.745
3,Oeste,2,184.990
4,Sur,1,230.000



--- 3. Promedio de Cantidad Vendida por Categoría ---


,Categoria,Promedio_Cantidad
0,Electrónica,2.333333
1,Hogar,2.000000
2,Moda,2.000000
3,No Especificado,2.000000
